# Fetching Data
- Includes all tickers even if they do not return data (prints those who fail)
- Only include FRQ=D (these Data Item Codes vary)
- Adjusted close price sets master date/ticker index (all sheets have the same time horizon and aount/order of tickers no matter the consistancy of datapoints) 

PS! Adjust time-horizon to one month max when applying new RICs/Tickers to test code compatibility 

    "Market_Cap2": "TR.F.MktCap(Period=FY0,Curn=EUR)",
    "Adjusted_Close": "TR.PriceClose(Curn=EUR)",
    "Price_Open": "TR.PriceOpen(Curn=EUR)"
    "Total_Return": "TR.TotalReturn1D",
    "Bid_Prices": "TR.BIDPRICE(Curn=EUR)",
    "Ask_Prices": "TR.ASKPRICE(Curn=EUR)",
    "Volume": "TR.Volume",
    "Turnover": "TR.Turnover(Curn=EUR)",
    "Shares_Outstanding": "TR.SharesOutstanding",
    "Market_Cap": "TR.CompanyMarketCap(Curn=EUR)",
    "Dividends": "TR.DPSActValue(Period=FQ0,Curn=EUR)"

Main RICs: 
- /Users/k.a.h/Master/Master2025/WholeNordic.txt ---> Includes all Nordic companies with atleast 6 months of consistent trading for any given time between 1989-2025. 




In [ ]:
import refinitiv.data as rd
import pandas as pd
import os
from datetime import datetime
import time
import numpy as np

# Opt-in to the future behavior to avoid the warning
pd.set_option('future.no_silent_downcasting', True)

# API Key
APP_KEY = "xxx"
BATCH_SIZE = 5

# Fields to fetch
FIELDS = {    
    "Market_Cap2": "TR.F.MktCap(Curn=EUR)",
    "Adjusted_Close": "TR.PriceClose(Curn=EUR)",
    "Price_Open": "TR.PriceOpen(Curn=EUR)"
    "Total_Return": "TR.TotalReturn1D",
    "Bid_Prices": "TR.BIDPRICE(Curn=EUR)",
    "Ask_Prices": "TR.ASKPRICE(Curn=EUR)",
    "Volume": "TR.Volume",
    "Turnover": "TR.Turnover(Curn=EUR)",
    "Shares_Outstanding": "TR.SharesOutstanding",
    "Market_Cap": "TR.CompanyMarketCap(Curn=EUR)",
    "Dividends": "TR.DPSActValue(Period=FQ0,Curn=EUR)"
}

# Open Refinitiv Session
def open_refinitiv_session():
    try:
        rd.open_session(app_key=APP_KEY)
        print("Refinitiv session opened successfully.")
    except Exception as e:
        print(f"Error opening session: {e}")

# Close Refinitiv Session
def close_refinitiv_session():
    try:
        rd.close_session()
        print("Refinitiv session closed.")
    except Exception as e:
        print(f"Error closing session: {e}")

# Read tickers from file
def read_tickers_from_file(filepath):
    try:
        with open(filepath, "r", encoding="utf-8") as file:
            tickers = [line.strip() for line in file.readlines() if line.strip() and line.strip() != "RIC"]
        print(f"Successfully read {len(tickers)} tickers.")
        return tickers
    except Exception as e:
        print(f"Error reading tickers: {e}")
        return []
    
# Set global limits
API_TIMEOUT = 120  # 2-minute timeout detection
MAX_RETRIES = 3    # Retry up to 3 times per batch
RETRY_DELAY = 5    # Wait 5 seconds before retrying
IDLE_THRESHOLD = 180  # 3 minutes of no output = force stop & restart API session


def fetch_field_data(field_name, refinitiv_field, tickers):
    """Fetches a single field for all tickers in batches, handling API stalls."""
    df_list = []
    failed_batches = []

    print(f"\nFetching {field_name} ({refinitiv_field}) for all tickers...")

    for i in range(0, len(tickers), BATCH_SIZE):
        batch = tickers[i:i + BATCH_SIZE]
        print(f"Fetching batch {i // BATCH_SIZE + 1} with {len(batch)} tickers...")
        batch_start_time = time.time()
        attempt = 0

        while attempt < MAX_RETRIES:
            try:
                # If batch is unresponsive, restart API session
                if time.time() - batch_start_time > IDLE_THRESHOLD:
                    print(f"WARNING: No response for {field_name} batch {i // BATCH_SIZE + 1} in {IDLE_THRESHOLD} sec. Restarting API session.")
                    close_refinitiv_session()
                    time.sleep(5)
                    open_refinitiv_session()
                    batch_start_time = time.time()

                # Fetch data. Remember to set start/end dates
                df = rd.get_history(
                    universe=batch,
                    fields=[refinitiv_field],
                    start="1989-12-01",
                    end="2025-02-13"
                )

                if not df.empty:
                    df.reset_index(inplace=True)
                    df.set_index("Date", inplace=True)
                    # Remove duplicate dates to ensure unique index
                    df = df.loc[~df.index.duplicated(keep="first")]
                    df_list.append(df)
                    print(f"Success: {field_name} batch {i // BATCH_SIZE + 1}")
                    break  # Stop retrying this batch
                else:
                    print(f"Warning: No data for {field_name} batch {i // BATCH_SIZE + 1}")
                    failed_batches.append(batch)
                    break

            except Exception as e:
                print(f"Error fetching {field_name} batch {i // BATCH_SIZE + 1} (Attempt {attempt + 1}/{MAX_RETRIES}): {e}")
                time.sleep(RETRY_DELAY)
                attempt += 1

        if attempt == MAX_RETRIES:
            print(f"ERROR: {field_name} batch {i // BATCH_SIZE + 1} failed after {MAX_RETRIES} attempts.")
            failed_batches.append(batch)

    # Concatenate batches ensuring no duplicate index values
    df_final = pd.concat(df_list, axis=1) if df_list else pd.DataFrame()
    for ticker in tickers:
        if ticker not in df_final.columns:
            df_final[ticker] = np.nan

    return df_final


def fetch_all_data(tickers):
    """Fetches each field separately for all tickers and aligns data properly."""
    dataframes = {}

    # Determine master field for date index
    potential_master_fields = ["TotalReturn"]  # Extend if needed
    master_field = None
    master_df = None

    for candidate in potential_master_fields:
        print(f"\nAttempting to set master date index from {candidate}...")
        master_df = fetch_field_data(candidate, FIELDS[candidate], tickers)
        master_df = master_df.loc[~master_df.index.duplicated(keep="first")]
        if not master_df.empty:
            master_field = candidate
            print(f"Master field selected: {master_field}")
            break
        else:
            print(f"Warning: {candidate} is empty. Trying next field...")

    if master_field is None:
        raise ValueError("All candidate master fields are empty. Cannot proceed.")

    master_dates = master_df.index.to_series().drop_duplicates().sort_values()
    fixed_ticker_order = sorted(master_df.columns)
    dataframes[master_field] = master_df[fixed_ticker_order]

    # Fetch remaining fields if any
    for field_name, refinitiv_field in FIELDS.items():
        if field_name == master_field:
            continue
        print(f"\nFetching {field_name} ({refinitiv_field}) for all tickers...")
        df = fetch_field_data(field_name, refinitiv_field, tickers)
        if df.empty:
            print(f"Skipping {field_name} due to missing data.")
            continue
        df = df.loc[~df.index.duplicated(keep="first")]
        df = df.reindex(master_dates)
        for ticker in tickers:
            if ticker not in df.columns:
                df[ticker] = np.nan
        df = df.loc[:, ~df.columns.duplicated()].reindex(columns=fixed_ticker_order)
        dataframes[field_name] = df

    save_to_excel_multi(dataframes)


def save_to_excel_multi(dataframes, filename="TotalReturn", folder="Nordics"):
    """Saves multiple DataFrames into separate sheets in an Excel file."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = os.path.join(folder, f"{filename}_{timestamp}.xlsx")
    os.makedirs(folder, exist_ok=True)

    try:
        with pd.ExcelWriter(file_path, engine="xlsxwriter") as writer:
            for field_name, df in dataframes.items():
                if not df.empty:
                    df.to_excel(writer, sheet_name=field_name[:31], index=True)
        print(f"All data saved in: {file_path}")
    except Exception as e:
        print(f"Error saving file: {e}")


if __name__ == "__main__":
    open_refinitiv_session()
    tickers = read_tickers_from_file("WholeNordic.txt")
    fetch_all_data(tickers)
    close_refinitiv_session()


# Fetch Fundamentals - Annual Frequency (These are not all variables included in the analysis)

In [ ]:
#!/usr/bin/env python
import sys
import numpy as np

# Ensure np.matrix is available (required by refinitiv.data)
if not hasattr(np, 'matrix'):
    np.matrix = np.array
    sys.modules['numpy'].matrix = np.array
    print("Patched np.matrix successfully at the top level.")

import refinitiv.data as rd
import pandas as pd
import os
from datetime import datetime
import time

# Opt-in to future behavior (optional)
pd.set_option('future.no_silent_downcasting', True)

# API Key
APP_KEY = "xxx"
BATCH_SIZE = 5

# Define fundamental fields for annual data in Euros.
# (Using (Period=FY0, Curn=EUR) to request annual fundamental data)
FIELDS = {
    "Revenue": "TR.Revenue(Period=FY0, Curn=EUR)",
    "Shares_Outstanding": "TR.F.ComShrOutsTot(Period=FY0)",
    "Gross_Profit": "TR.GrossProfit(Period=FY0, Curn=EUR)",
    "EBITDA": "TR.EBITDA(Period=FY0, Curn=EUR)",
    "EBIT": "TR.EBIT(Period=FY0, Curn=EUR)",
    "Income_Before_Taxes": "TR.F.IncBefTax(Period=FY0, Curn=EUR)",
    "Net_Income_Actual": "TR.NetProfitActValue(Period=FY0, Curn=EUR)",
    "Retained_Earnings_Total": "TR.F.RetainedEarnTot(Period=FY0, Curn=EUR)",
    "Total_Current_Assets": "TR.F.TotCurrAssets(Period=FY0, Curn=EUR)",
    "Total_Current_Liabilities": "TR.F.TotCurrLiab(Period=FY0, Curn=EUR)",
    "Total_Debt": "TR.TotalDebt(Period=FY0, Curn=EUR)",
    "Total_Liabilities": "TR.TotalLiabilities(Period=FY0, Curn=EUR)",
    "Working_Capital": "TR.F.WkgCap(Period=FY0, Curn=EUR)",
    "Working_Capital_NonCash": "TR.F.WkgCapNonCash(Period=FY0, Curn=EUR)",
    "Net_Cash_Flow_Operating": "TR.F.NetCashFlowOp(Period=FY0, Curn=EUR)",
    "Depreciation_Total": "TR.F.DeprTot(Period=FY0, Curn=EUR)",
    "Market_Capitalization": "TR.F.MktCap(Period=FY0, Curn=EUR)",
    "Total_Assets": "TR.TotalAssets(Period=FY0, Curn=EUR)",
    "Net_Income_After_Minority": "TR.F.NetIncAfterMinIntr(Period=FY0, Curn=EUR)",
    "Capital_Expenditures_Total": "TR.F.CAPEXTot(Period=FY0, Curn=EUR)",
    "Common_Equity_Total": "TR.F.ComEqTot(Period=FY0, Curn=EUR)"
}

# Set the date range for fetching data
START_DATE = "1998-12-01"
END_DATE   = "2025-02-13"

def read_tickers_from_file(filepath):
    try:
        with open(filepath, "r", encoding="utf-8") as file:
            tickers = [line.strip() for line in file.readlines() if line.strip() and line.strip() != "RIC"]
        print(f"Successfully read {len(tickers)} tickers.")
        return tickers
    except Exception as e:
        print(f"Error reading tickers: {e}")
        return []

def open_refinitiv_session():
    try:
        rd.open_session(app_key=APP_KEY)
        print("Refinitiv session opened successfully.")
    except Exception as e:
        print(f"Error opening session: {e}")

def close_refinitiv_session():
    try:
        rd.close_session()
        print("Refinitiv session closed.")
    except Exception as e:
        print(f"Error closing session: {e}")

def snap_to_year_end(df, ticker_order):
    """
    Adjusts a wide DataFrame (columns=tickers, index=Date) so that
    for each ticker, the data is snapped to the annual date in 'yyyy-12' format.
    
    The function does this by:
      1. Resetting the index to convert Date from the index to a column.
      2. Melting the DataFrame into long format.
      3. Creating a new column 'YearEnd' formatted as 'yyyy-12'.
      4. Grouping by [Ticker, YearEnd] and selecting the row with the latest actual date.
      5. Pivoting back to wide format and reordering columns per ticker_order.
    """
    if df.empty:
        return df

    # 1. Convert the index to a column (Date)
    df_long = df.reset_index().melt(id_vars="Date", var_name="Ticker", value_name="Value")
    
    # 2. Create the YearEnd column in the format 'yyyy-12' (using December to represent the year)
    df_long["YearEnd"] = df_long["Date"].dt.year.astype(str) + "-12"
    
    # 3. For each group (Ticker, YearEnd), pick the row with the latest actual date
    def pick_last_in_year(g):
        return g.loc[g["Date"].idxmax()]
    
    df_snapped = df_long.groupby(["Ticker", "YearEnd"], as_index=False).apply(pick_last_in_year).reset_index(drop=True)
    
    # 4. Rename 'YearEnd' to 'Date' (this will serve as the common annual index) 
    df_snapped.rename(columns={"YearEnd": "Date"}, inplace=True)
    # Drop the original Date column (the detailed actual date) if not needed
    df_snapped.drop(columns=["Date"], inplace=True, errors='ignore')
    
    # Instead, we now want to use the snapped date from the 'Date' column created above.
    # Since we dropped the original 'Date', we re-add it from our YearEnd values:
    # (Alternatively, we could have simply overwritten 'Date'; here we mimic the pattern.)
    df_snapped["Date"] = df_long.groupby(["Ticker", "YearEnd"]).apply(lambda g: g["YearEnd"].iloc[0]).values
    
    # 5. Set the 'Date' as index
    df_snapped.set_index("Date", inplace=True)
    # 6. Pivot back to wide format: index=snapped annual date, columns=tickers
    df_wide = df_snapped.pivot(columns="Ticker", values="Value")
    # 7. Reindex columns to preserve the original ticker order
    df_wide = df_wide.reindex(columns=ticker_order)
    return df_wide

# Batch fetch a single field using get_history() then snap data to annual dates.
def fetch_field_data(field_name, refinitiv_field, tickers):
    df_list = []
    failed_batches = []
    print(f"\nFetching {field_name} ({refinitiv_field}) for all tickers...")
    for i in range(0, len(tickers), BATCH_SIZE):
        batch = tickers[i:i+BATCH_SIZE]
        print(f"Fetching batch {i//BATCH_SIZE+1} with {len(batch)} tickers...")
        attempt = 0
        while attempt < MAX_RETRIES:
            try:
                # Fetch data with monthly interval
                df = rd.get_history(
                    universe=batch,
                    fields=[refinitiv_field],
                    start=START_DATE,
                    end=END_DATE,
                    interval="1M"
                )
                if not df.empty:
                    df.reset_index(inplace=True)
                    df.set_index("Date", inplace=True)
                    # Remove duplicate dates in this batch
                    df = df[~df.index.duplicated(keep="first")]
                    df_list.append(df)
                    print(f"Success: {field_name} batch {i//BATCH_SIZE+1}")
                    break
                else:
                    print(f"Warning: No data for {field_name} batch {i//BATCH_SIZE+1}")
                    failed_batches.append(batch)
                    break
            except Exception as e:
                print(f"Error fetching {field_name} batch {i//BATCH_SIZE+1} (Attempt {attempt+1}/{MAX_RETRIES}): {e}")
                time.sleep(RETRY_DELAY)
                attempt += 1
        if attempt == MAX_RETRIES:
            print(f"ERROR: {field_name} batch {i//BATCH_SIZE+1} failed after {MAX_RETRIES} attempts.")
            failed_batches.append(batch)
    df_final = pd.concat(df_list, axis=1) if df_list else pd.DataFrame()
    # Remove duplicate index entries in final DataFrame
    if not df_final.empty:
        df_final = df_final[~df_final.index.duplicated(keep="first")]
    # Ensure every ticker is present as a column.
    for ticker in tickers:
        if ticker not in df_final.columns:
            df_final[ticker] = np.nan

    # Snap monthly data to annual dates (format: yyyy-12)
    df_final = snap_to_year_end(df_final, tickers)
    return df_final

# Fetch all fields and align them on a common annual date index from the master field.
def fetch_all_data(tickers):
    dataframes = {}
    potential_master_fields = ["Revenue", "Gross_Profit", "EBITDA", "EBIT", "Income_Before_Taxes"]
    master_field = None
    master_df = None
    for candidate in potential_master_fields:
        print(f"\nAttempting to set master date index from {candidate}...")
        master_df = fetch_field_data(candidate, FIELDS[candidate], tickers)
        if not master_df.empty:
            master_df = master_df.loc[~master_df.index.duplicated(keep="first")]
            master_field = candidate
            print(f"Master field selected: {master_field}")
            break
        else:
            print(f"Warning: {candidate} is empty. Trying next field...")
    if master_field is None:
        raise ValueError("All candidate master fields are empty. Cannot proceed.")
    master_dates = master_df.index.drop_duplicates().sort_values()
    fixed_ticker_order = sorted(master_df.columns)
    dataframes[master_field] = master_df[fixed_ticker_order]
    for field_name, refinitiv_field in FIELDS.items():
        if field_name == master_field:
            continue
        print(f"\nFetching {field_name} ({refinitiv_field}) for all tickers...")
        df = fetch_field_data(field_name, refinitiv_field, tickers)
        if df.empty:
            print(f"Skipping {field_name} due to missing data.")
            continue
        df = df.loc[~df.index.duplicated(keep="first")]
        # Reindex to the master annual dates
        df = df.reindex(master_dates)
        for ticker in tickers:
            if ticker not in df.columns:
                df[ticker] = np.nan
        df = df.loc[:, ~df.columns.duplicated()].reindex(columns=fixed_ticker_order)
        dataframes[field_name] = df
    return dataframes

# Save the fetched data into an Excel file.
def save_to_excel_multi(dataframes, filename="/Users/k.a.h/Master/Master2025/Quarterly/NORannual.xlsx"):
    try:
        with pd.ExcelWriter(filename, engine="xlsxwriter") as writer:
            for field_name, df in dataframes.items():
                if not df.empty:
                    df.to_excel(writer, sheet_name=field_name[:31], index=True)
        print(f"All data saved in: {filename}")
    except Exception as e:
        print(f"Error saving file: {e}")

# Global retry and delay settings
API_TIMEOUT = 120
MAX_RETRIES = 3
RETRY_DELAY = 5
IDLE_THRESHOLD = 180

# Main execution
if __name__ == "__main__":
    open_refinitiv_session()
    tickers = read_tickers_from_file("norwegian_tickers.txt")
    dataframes = fetch_all_data(tickers)
    save_to_excel_multi(dataframes, filename="/Users/k.a.h/Master/Master2025/Quarterly/NORannual.xlsx")
    close_refinitiv_session()


# Fetch Fundamentals - Quarterly
PS! Some tickers might disrupt sheet layout in output file due to inconsistent fundamental coverage (both Refinitiv and firms themselves)

In [ ]:
#!/usr/bin/env python
import sys
import numpy as np

# Ensure np.matrix is available (required by refinitiv.data)
if not hasattr(np, 'matrix'):
    np.matrix = np.array
    sys.modules['numpy'].matrix = np.array
    print("Patched np.matrix successfully at the top level.")

import refinitiv.data as rd
import pandas as pd
import os
import time

# Opt-in to future behavior (optional)
pd.set_option('future.no_silent_downcasting', True)

# API Key
APP_KEY = "xxx"
BATCH_SIZE = 5
MAX_RETRIES = 3
RETRY_DELAY = 5

# Define fundamental fields for quarterly data in Euros (Period=FY0, Curn=EUR)
FIELDS = {
    "Dep&Amortization": "TR.DepreciationAmortizationActValue(Period=FQ0,Curn=EUR)",
}

# Date range (fetch monthly data, then snap to quarter-end)
START_DATE = "1998-12-01"
END_DATE   = "2025-02-13"

def read_tickers_from_file(filepath):
    try:
        with open(filepath, "r", encoding="utf-8") as file:
            # Read tickers preserving order and ignoring header "RIC"
            tickers = [line.strip() for line in file if line.strip() and line.strip() != "RIC"]
        print(f"Successfully read {len(tickers)} tickers.")
        return tickers
    except Exception as e:
        print(f"Error reading tickers: {e}")
        return []

def open_refinitiv_session():
    try:
        rd.open_session(app_key=APP_KEY)
        print("Refinitiv session opened successfully.")
    except Exception as e:
        print(f"Error opening session: {e}")

def close_refinitiv_session():
    try:
        rd.close_session()
        print("Refinitiv session closed.")
    except Exception as e:
        print(f"Error closing session: {e}")

def snap_to_quarter_end(df, ticker_order):
    """
    Takes a wide DataFrame with columns = tickers, monthly/daily DateTimeIndex,
    and returns a new DataFrame with one row per (Ticker, Quarter),
    labeled on the official quarter-end date (03/31, 06/30, 09/30, 12/31).

    Each ticker's last data point in the quarter is relabeled to that quarter-end.
    """
    if df.empty:
        return df

    # 1) Melt the DataFrame into long format: columns = [Date, Ticker, Value]
    df_long = df.reset_index().melt(id_vars="Date", var_name="Ticker", value_name="Value")

    # 2) For each row, find the official quarter-end date
    #    e.g. 2005-03-30 -> 2005Q1 -> 2005-03-31
    df_long["QuarterEnd"] = df_long["Date"].dt.to_period("Q").dt.end_time

    # 3) Group by (Ticker, QuarterEnd) and pick the row with the largest "Date" in that group
    def pick_last_in_quarter(g):
        return g.loc[g["Date"].idxmax()]

    df_snapped = (
        df_long.groupby(["Ticker", "QuarterEnd"], as_index=False)
               .apply(pick_last_in_quarter)
               .reset_index(drop=True)
    )

    # 4) Rename "QuarterEnd" -> "QtrEndDate" and set it as the index.
    df_snapped.rename(columns={"QuarterEnd": "QtrEndDate"}, inplace=True)
    df_snapped.set_index("QtrEndDate", inplace=True)
    df_snapped.index.name = "Date"
    df_snapped.drop(columns=["Date"], inplace=True)

    # 5) Pivot back to wide form: index=quarter-end date, columns=tickers
    df_wide = df_snapped.pivot(columns="Ticker", values="Value")

    # 6) Reindex columns to preserve the original ticker order
    df_wide = df_wide.reindex(columns=ticker_order)

    return df_wide

def fetch_field_data(field_name, refinitiv_field, tickers):
    """Fetch monthly data for a single field, then snap each ticker to quarter-end."""
    df_list = []
    print(f"\nFetching {field_name} ({refinitiv_field}) for all tickers...")

    for i in range(0, len(tickers), BATCH_SIZE):
        batch = tickers[i:i+BATCH_SIZE]
        print(f"Fetching batch {i//BATCH_SIZE+1} with {len(batch)} tickers...")
        attempt = 0

        while attempt < MAX_RETRIES:
            try:
                # Use interval="1M" for monthly data
                df = rd.get_history(
                    universe=batch,
                    fields=[refinitiv_field],
                    start=START_DATE,
                    end=END_DATE,
                    interval="1M"
                )
                if not df.empty:
                    df.reset_index(inplace=True)
                    df.set_index("Date", inplace=True)
                    # Remove duplicate date rows
                    df = df[~df.index.duplicated(keep="first")]
                    df_list.append(df)
                    print(f"Success: {field_name} batch {i//BATCH_SIZE+1}")
                    break
                else:
                    print(f"Warning: No data for {field_name} batch {i//BATCH_SIZE+1}")
                    break
            except Exception as e:
                print(f"Error fetching {field_name} batch {i//BATCH_SIZE+1} (Attempt {attempt+1}/{MAX_RETRIES}): {e}")
                time.sleep(RETRY_DELAY)
                attempt += 1

    # Concatenate all batches for this field
    df_final = pd.concat(df_list, axis=1) if df_list else pd.DataFrame()
    # Remove duplicates in the final DataFrame
    if not df_final.empty:
        df_final = df_final[~df_final.index.duplicated(keep="first")]

    # Ensure every ticker is present as a column.
    # If a ticker is missing, add it as a column with NaN values.
    for ticker in tickers:
        if ticker not in df_final.columns:
            df_final[ticker] = np.nan

    # --- NEW ORDERING STEP ---
    # Reindex the columns according to the original order from the tickers file.
    df_final = df_final.reindex(columns=tickers)

    # Snap each ticker's monthly data to the official quarter-end date
    df_final = snap_to_quarter_end(df_final, tickers)

    return df_final

def fetch_all_data(tickers):
    """
    Fetches data for each field, storing them in a dict: field_name -> DataFrame (quarter-end index).
    We do NOT unify the date index across fields. Each field has its own quarter-end index.
    """
    dataframes = {}
    for field_name, refinitiv_field in FIELDS.items():
        df = fetch_field_data(field_name, refinitiv_field, tickers)
        if df.empty:
            print(f"Skipping {field_name} due to no data.")
            continue
        dataframes[field_name] = df
    return dataframes

def save_to_excel_multi(dataframes, filename="/Users/k.a.h/Master/Master2025/Quarterly/ICEshares.xlsx"):
    try:
        with pd.ExcelWriter(filename, engine="xlsxwriter") as writer:
            for field_name, df in dataframes.items():
                if not df.empty:
                    df.to_excel(writer, sheet_name=field_name[:31], index=True)
        print(f"All data saved in: {filename}")
    except Exception as e:
        print(f"Error saving file: {e}")

# Main
if __name__ == "__main__":
    # Retry / idle thresholds (unused in minimal code, but kept for reference)
    API_TIMEOUT = 120
    IDLE_THRESHOLD = 180

    open_refinitiv_session()
    tickers = read_tickers_from_file("swedish_tickers_noD.txt")
    dataframes = fetch_all_data(tickers)
    save_to_excel_multi(dataframes, filename="/Users/k.a.h/Master/Master2025/Quarterly/D&ASWEQuarterly.xlsx")
    close_refinitiv_session()


# Benchmarks

In [ ]:
import pandas as pd
import refinitiv.data as rd

# Set up API Key
APP_KEY = "e558a80ac54f413c9d70be237c6c63c7221207ec"
rd.open_session(app_key=APP_KEY)

# Define benchmark indices
benchmark_indices = {
    "Norway": ".OSEBX",
    "Sweden": ".OMXS30",
    "Finland": ".OMXH25",
    "Denmark": ".OMXC20",
    "Iceland": ".OMXIPI"
}

# Define date range
start_date = "1989-12-01"
end_date = "2025-02-13"

# Iterate through each index
for country, ric in benchmark_indices.items():
    try:
        # Fetch historical data (interpolated)
        df = rd.get_history(
            universe=[ric],
            fields=["TR.PriceClose(Curn=EUR)"],
            start=start_date,
            end=end_date
        )

        if not df.empty:
            # Interpolate missing values
            df["Price Close"] = df["Price Close"].interpolate(method="linear")

            # Calculate "parameter" total return
            df["Total Return"] = df["Price Close"].pct_change()

            # Define output path
            output_path = f"/Users/k.a.h/Master/Master2025/Index/{country}_Total_Yearly_Return.xlsx"

            # Save to Excel with separate sheets
            with pd.ExcelWriter(output_path) as writer:
                df[["Price Close"]].to_excel(writer, sheet_name="Price_Close_EUR")
                df[["Total Return"]].to_excel(writer, sheet_name="Total_Return_EUR")

            print(f"Successfully saved total return and price close data for {country} at {output_path}")
        else:
            print(f"No data available for {country}")

    except Exception as e:
        print(f"Failed to calculate total return for {country}: {e}")

rd.close_session()


# Industry and Sector

In [ ]:
import refinitiv.data as rd
import pandas as pd

# Set up the API Key
APP_KEY = "e558a80ac54f413c9d70be237c6c63c7221207ec"
rd.open_session(app_key=APP_KEY)

# Define benchmark indices for each country
benchmark_indices = {
    "Norway": ".OSEBX",
    "Sweden": ".OMXS30",
    "Finland": ".OMXH25",
    "Denmark": ".OMXC20",
    "Iceland": ".OMXIPI"
}

# Define the date range
start_date = "1989-12-01"
end_date = "2025-02-19"

# Fetch benchmark data for each country
for country, ric in benchmark_indices.items():
    try:
        df = rd.get_history(
            universe=[ric],
            fields=["TR.PriceClose(Curn=EUR)"],
            start=start_date,
            end=end_date
        )

        if not df.empty:
            # Interpolate missing values using linear interpolation
            df = df.interpolate(method='linear')

            # Define output file for the country
            output_path = f"/Users/k.a.h/Master/Master2025/{country}_Benchmark_EUR.xlsx"

            # Save each field in a separate sheet
            with pd.ExcelWriter(output_path) as writer:
                df["Price Close"].to_excel(writer, sheet_name="PriceClose_EUR")

            print(f"Successfully saved {country} benchmark data to {output_path}")
        else:
            print(f"No benchmark data available for {country}")

    except Exception as e:
        print(f"Failed to fetch benchmark data for {country}: {e}")

rd.close_session()


# Check Adjustment Features of Data Item Codes 

In [ ]:
import refinitiv.data as rd
import pandas as pd

APP_KEY = "e558a80ac54f413c9d70be237c6c63c7221207ec"
rd.open_session(app_key=APP_KEY)

history_df = rd.get_history(
    universe=['EQNR.OL'],
    fields=['TR.CompanyMarketCapitalization'],
    adjustments=['exchangeCorrection', 'manualCorrection', 'CCH', 'CRE', 'RPO', 'RTS'],
    start='2022-01-01',
    end='2025-02-07'
)

print(history_df)

rd.close_session()

# Check time-horizon before fetching
- Useful when fetching data for long periods

In [ ]:
import refinitiv.data as rd
import pandas as pd

APP_KEY = "e558a80ac54f413c9d70be237c6c63c7221207ec"
rd.open_session(app_key=APP_KEY)

# Define OSEBX index
osebx_ric = ".OMXIPI"

# Function to find the earliest available data
def find_earliest_available_date():
    start_year = 2025  # Start from today
    step = -5  # Move backwards in 5-year steps
    
    while start_year > 1960:  # Stop at 1960 to avoid unnecessary API calls
        try:
            history_df = rd.get_history(
                universe=[osebx_ric],
                fields=["TR.CLOSEPRICE"],
                start=f"{start_year}-01-01",
                end=f"{start_year}-01-10"
            )
            if not history_df.empty:
                print(f"Data available from {start_year}")
            else:
                print(f"No data found for {start_year}")
                return start_year + 5  # Last known good year
        except Exception as e:
            print(f"Error fetching data for {start_year}: {e}")
            return start_year + 5  # Return the last working year

        start_year += step  # Move further back

    return None

# Run the search
earliest_date = find_earliest_available_date()
print(f"Earliest available data starts from: {earliest_date}")

rd.close_session()
